# Actividad 05 - 03. GraphCodeBERT (modelo de lenguaje) en la GPU del Mac

**NewCodeCreds** · Edward Alexander Gómez Ispanel, Jocelyn Abigail De León Ardón y Brazil Abraham
Batres Cruz - Maestría en Data Science - Investigación en Data Science - Noviembre 2026

Datos: la tabla de análisis del notebook 01 (`datos_procesados/lineas.parquet`) y los archivos de
`CredData/data`, de donde se toma la línea candidata y sus líneas vecinas.

**Este notebook se corre en el Mac M5**, no en la máquina virtual: usa la GPU de Apple
mediante el backend `mps` de PyTorch. Si `mps` no está disponible, cae a CPU (muy lento).

Requisitos (una sola vez, en el entorno de Jupyter del Mac):

`pip install torch transformers pandas pyarrow scikit-learn`

Qué hace: ajusta `microsoft/graphcodebert-base` como clasificador binario de la línea
candidata (T frente a F/X), con la línea y una línea de contexto antes y después, en los
mismos pliegues que Random Forest. Guarda `datos_procesados/predicciones_graphcodebert.parquet`,
que el notebook 04 incorpora automáticamente.

Para controlar el tiempo, `MODO = "rapido"` ajusta solo el esquema aleatorio deduplicado y el
de organización (10 ajustes, 1 época, con submuestreo del entrenamiento). `MODO = "completo"`
agrega los ocho tipos de archivo y 2 épocas.

**Sobre las salidas guardadas.** Las salidas de este notebook son las de la corrida en modo
rápido del 14 de noviembre de 2026 (Mac M5 Pro, unos 6 minutos por ajuste). Esa corrida usó la
versión anterior del código, con dos defectos que ya están corregidos abajo para la corrida
completa y que se declaran como limitación de los resultados del modo rápido:

1. Las líneas se leían con `str.splitlines()`, que corta en caracteres como `\x0c` y desalinea el
   número de línea en 5 archivos (35 líneas candidatas, 8 T). Ahora se usa la regla de CredData
   (`ids_utils.lineas_de_archivo`).
2. La entrada era `línea anterior + línea candidata + línea siguiente` truncada a 128 tokens,
   con el contexto previo primero. En el conjunto deduplicado la línea candidata quedaba
   cortada en 5,529 líneas (11.1%; 2,201 T) y completamente fuera de la ventana en 192.
   Ahora la línea candidata va como primer segmento y el contexto como segundo, y en modo
   completo `MAX_LEN` sube a 256: quedan cortadas 1,179 líneas, todas porque la línea sola
   supera los 256 tokens.

La corrida rápida usó la tabla de la versión anterior del notebook 01, resumida sin texto en
`datos_procesados/graphcodebert_tabla_v1.parquet`.
Los pliegues son los mismos que usa Random Forest, porque el notebook 01 ahora lee esas
asignaciones de `resultados/asignacion_pliegues_*.csv`. El notebook 04 evalúa GraphCodeBERT solo
en las líneas que coinciden entre las dos versiones (misma etiqueta y mismo texto).

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers.utils import logging as hf_logging

sys.path.append(".")
from ids_utils import SEMILLA, lineas_de_archivo, metricas

# Sin el reporte de pesos ("LOAD REPORT") ni la barra de progreso en cada pliegue
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

MODO = "rapido"          # "rapido" o "completo"
MAX_TRAIN = 20000        # submuestreo estratificado del entrenamiento en modo rápido
MAX_LEN = 128 if MODO == "rapido" else 256
BATCH = 32
LR = 2e-5
EPOCAS = 1 if MODO == "rapido" else 2
MODELO = "microsoft/graphcodebert-base"

dispositivo = "mps" if torch.backends.mps.is_available() else "cpu"
print("Dispositivo:", dispositivo)
torch.manual_seed(SEMILLA)

CRED = Path("../../CredData")
DATOS = Path("../datos_procesados")
lineas = pd.read_parquet(DATOS / "lineas.parquet")

Dispositivo: mps


## 1. Texto de entrada: línea candidata con una línea de contexto

La línea candidata es el primer segmento y el contexto (línea anterior y siguiente) el segundo.
Con `truncation="longest_first"` se recorta primero el segmento más largo, así que el contexto
no desplaza a la línea candidata fuera de la ventana.

In [ ]:
cache = {}


def partes(fp, ls, le):
    if fp not in cache:
        cache[fp] = lineas_de_archivo((CRED / fp).read_text(encoding="utf-8", errors="ignore"))
    t = cache[fp]
    antes = t[ls - 2] if ls >= 2 else ""
    despues = t[le] if le < len(t) else ""
    centro = "\n".join(t[ls - 1:le])
    return centro[:600], f"{antes[:200]}\n{despues[:200]}"


lineas[["entrada", "contexto"]] = pd.DataFrame(
    [partes(*x) for x in zip(lineas.FilePath, lineas.LineStart, lineas.LineEnd)], index=lineas.index)
cache.clear()
tok = AutoTokenizer.from_pretrained(MODELO)
n_tok = np.array([len(x) for x in tok(list(lineas.entrada[lineas.dedup == 1]), add_special_tokens=False)["input_ids"]])
print(f"Líneas candidatas que no caben solas en {MAX_LEN} tokens: {(n_tok > MAX_LEN - 4).sum()} "
      f"({(n_tok > MAX_LEN - 4).mean():.1%})")


class Lineas(Dataset):
    def __init__(self, idx, y):
        self.enc = tok(list(lineas.entrada.iloc[idx]), list(lineas.contexto.iloc[idx]),
                       truncation="longest_first", max_length=MAX_LEN, padding="max_length",
                       return_tensors="pt")
        self.y = torch.tensor(np.asarray(y), dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.y[i]


def ajustar_y_predecir(idx_tr, idx_te, y):
    rng = np.random.default_rng(SEMILLA)
    if MODO == "rapido" and len(idx_tr) > MAX_TRAIN:
        pos = idx_tr[y[idx_tr] == 1]
        neg = idx_tr[y[idx_tr] == 0]
        k_pos = int(MAX_TRAIN * len(pos) / len(idx_tr))
        idx_tr = np.concatenate([rng.choice(pos, k_pos, replace=False),
                                 rng.choice(neg, MAX_TRAIN - k_pos, replace=False)])
    modelo = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2).to(dispositivo)
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR)
    peso = torch.tensor([1.0, (y[idx_tr] == 0).sum() / max((y[idx_tr] == 1).sum(), 1)],
                        dtype=torch.float32, device=dispositivo)
    perdida = torch.nn.CrossEntropyLoss(weight=peso)
    dl = DataLoader(Lineas(idx_tr, y[idx_tr]), batch_size=BATCH, shuffle=True)
    modelo.train()
    for _ in range(EPOCAS):
        for xb, yb in dl:
            xb = {k: v.to(dispositivo) for k, v in xb.items()}
            out = modelo(**xb).logits
            loss = perdida(out, yb.to(dispositivo))
            opt.zero_grad()
            loss.backward()
            opt.step()
    modelo.eval()
    probs = []
    with torch.no_grad():
        for xb, _ in DataLoader(Lineas(idx_te, y[idx_te]), batch_size=128):
            xb = {k: v.to(dispositivo) for k, v in xb.items()}
            probs.append(torch.softmax(modelo(**xb).logits, -1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

## 2. Ajuste en los mismos pliegues que Random Forest

In [ ]:
esquemas = {"aleatorio": "pliegue_aleatorio", "organizacion": "pliegue_organizacion"}
if MODO == "completo":
    esquemas["tipo"] = "pliegue_tipo"

y = lineas.y_dedup.fillna(0).to_numpy().astype(int)
universo = lineas.index[lineas.dedup == 1].to_numpy()
res = lineas[["id_linea"]].copy()
for nombre, col in esquemas.items():
    res[f"score_gcb_{nombre}"] = np.nan
    pliegues = [p for p in lineas[col].unique() if p not in ("", -1)]
    for p in sorted(pliegues, key=str):
        t0 = time.time()
        idx_te = lineas.index[lineas[col] == p].to_numpy()
        quitar = lineas.index[lineas.tipo_archivo == p] if nombre == "tipo" else idx_te
        idx_tr = np.setdiff1d(universo, quitar)
        res.loc[idx_te, f"score_gcb_{nombre}"] = ajustar_y_predecir(idx_tr, idx_te, y)
        print(nombre, p, f"{time.time() - t0:.0f} s")
        res.to_parquet(DATOS / "predicciones_graphcodebert.parquet", index=False)  # guarda avance
    res[f"pred_gcb_{nombre}"] = (res[f"score_gcb_{nombre}"] >= 0.5).astype("Int64").where(
        res[f"score_gcb_{nombre}"].notna())

res.to_parquet(DATOS / "predicciones_graphcodebert.parquet", index=False)
res.describe().T

aleatorio 0 342 s
aleatorio 1 349 s
aleatorio 2 348 s
aleatorio 3 351 s
aleatorio 4 351 s
organizacion 0 375 s
organizacion 1 345 s
organizacion 2 354 s
organizacion 3 345 s
organizacion 4 328 s


,count,mean,std,min,25%,50%,75%,max
id_linea,64797.0,32398.0,18705.427033,0.0,16199.0,32398.0,48597.0,64796.0
score_gcb_aleatorio,49624.0,0.322882,0.454299,0.000292,0.000795,0.001921,0.982842,0.999258
pred_gcb_aleatorio,49624.0,0.324178,0.468072,0.0,0.0,0.0,1.0,1.0
score_gcb_organizacion,49624.0,0.305812,0.415412,0.000286,0.001026,0.004918,0.744628,0.999013
pred_gcb_organizacion,49624.0,0.334979,0.471988,0.0,0.0,0.0,1.0,1.0


## 3. Resumen: F1 y MCC por partición

Sobre las líneas deduplicadas, con umbral 0.5. Es solo un control de la corrida: las
diferencias pareadas con intervalo de confianza se calculan en el notebook 04.

In [ ]:
d = lineas[lineas.dedup == 1].join(res.drop(columns="id_linea"))
filas = []
for nombre in esquemas:
    g = d.dropna(subset=[f"score_gcb_{nombre}"])
    m = metricas(g.y_dedup.astype(int), (g[f"score_gcb_{nombre}"] >= 0.5).astype(int), g[f"score_gcb_{nombre}"])
    filas.append({"particion": nombre, **m})
pd.DataFrame(filas).round(3)

,particion,n,positivos,precision,recall,f1,mcc,fuga_residual,pr_auc
0,aleatorio,49624,15205,0.932,0.986,0.958,0.940,0.014,0.984
1,organizacion,49624,15205,0.885,0.968,0.925,0.891,0.032,0.939


**Qué salió.** En modo rápido, GraphCodeBERT pasa de F1 0.958 en la partición aleatoria a 0.925
en organizaciones no vistas (−0.033), y MCC de 0.940 a 0.891 (−0.049). PR-AUC baja de 0.984 a
0.939. La caída es menor que la de Random Forest y tiene otra forma: el recall casi no cambia
(0.986 a 0.968; la fuga residual pasa de 1.4% a 3.2%), mientras que la precisión baja de 0.932 a
0.885, así que en código de organizaciones nuevas el modelo genera más falsas alarmas. Las
diferencias pareadas con intervalo de confianza están en el notebook 04. Son resultados del modo
rápido (20,000 líneas de entrenamiento, 1 época) con los dos defectos descritos al inicio.